<a href="https://colab.research.google.com/github/saketh-varma06/DL_ASSIGNMENTS/blob/main/AI_DL_ASS_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install -q torch torchvision scikit-learn matplotlib seaborn

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
# Import libraries
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cpu


In [ ]:
!kaggle competitions download -c dogs-vs-cats
!unzip -q dogs-vs-cats.zip -d /content/
!mkdir -p /content/train
!unzip -q /content/train.zip -d /content/train/
!mkdir -p /content/test1
!unzip -q /content/test1.zip -d /content/test1/

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
Unzipping dogs-vs-cats.zip...
unzip:  cannot find or open dogs-vs-cats.zip, dogs-vs-cats.zip.zip or dogs-vs-cats.zip.ZIP.
Unzipping train.zip to /content/train/...
unzip:  cannot find or open /content/train.zip, /content/train.zip.zip or /content/train.zip.ZIP.
Unzipping test1.zip to /content/test1/...
unzip:  cannot find or open /content/test1.zip, /content/test1.zip.zip or /content/test1.zip.ZIP.
Verifying content of /content/train:
/content/train:

Verifying content of /content/test1:
/content/test1:


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

In [ ]:
import shutil
from PIL import Image

base_train_dir = '/content/train'
cat_dir = os.path.join(base_train_dir, 'Cat')
dog_dir = os.path.join(base_train_dir, 'Dog')

os.makedirs(cat_dir, exist_ok=True)
os.makedirs(dog_dir, exist_ok=True)

for filename in os.listdir(base_train_dir):
    if filename.endswith('.jpg'):
        if filename.startswith('cat'):
            src_path = os.path.join(base_train_dir, filename)
            dst_path = os.path.join(cat_dir, filename)
            shutil.move(src_path, dst_path)
        elif filename.startswith('dog'):
            src_path = os.path.join(base_train_dir, filename)
            dst_path = os.path.join(dog_dir, filename)
            shutil.move(src_path, dst_path)

for folder in [cat_dir, dog_dir]:
    for img_name in os.listdir(folder):
        img_path = os.path.join(folder, img_name)
        if img_name == 'Thumbs.db':
            os.remove(img_path)
            continue
        try:
            img = Image.open(img_path)
            img.verify()
        except Exception as e:
            os.remove(img_path)